In [1]:
import pandas as pd
from rapidfuzz.distance import Levenshtein
from tqdm import tqdm

In [2]:
df = pd.read_csv("./Generate_positions/generated_puzzles.csv")
df_lichess = pd.read_csv("./Generate_positions/generated_puzzles_lichess_dataset_theme_distribution.csv")
df_lichess_large = pd.read_csv("./Generate_positions/generated_puzzles_lichess_dataset_theme_distribution_large.csv")
df_lichess_correct_ratings = pd.read_csv("./Generate_positions/generated_puzzles_lichess_dataset_theme_distribution_correct_ratings.csv")

In [3]:
print(df["is_legal"].mean(), df[df["is_legal"]]["is_puzzle"].mean(), df[df["is_legal"] & df["is_puzzle"]]["counter_intuitive"].mean())
print(df_lichess["is_legal"].mean(), df_lichess[df_lichess["is_legal"]]["is_puzzle"].mean(), df_lichess[df_lichess["is_legal"] & df_lichess["is_puzzle"]]["counter_intuitive"].mean())
print(df_lichess_large["is_legal"].mean(), df_lichess_large[df_lichess_large["is_legal"]]["is_puzzle"].mean(), df_lichess_large[df_lichess_large["is_legal"] & df_lichess_large["is_puzzle"]]["counter_intuitive"].mean())
print(df_lichess_correct_ratings["is_legal"].mean(), df_lichess_correct_ratings[df_lichess_correct_ratings["is_legal"]]["is_puzzle"].mean(), df_lichess_correct_ratings[df_lichess_correct_ratings["is_legal"] & df_lichess_correct_ratings["is_puzzle"]]["counter_intuitive"].mean())

0.965 0.061139896373056994 0.01694915254237288
0.955 0.2094240837696335 0.01
0.9656 0.21644573322286662 0.0028708133971291866
0.973457532051282 0.1761498096511987 0.027453271028037383


In [6]:
(df_lichess_correct_ratings["is_legal"] & df_lichess_correct_ratings["is_puzzle"] & df_lichess_correct_ratings["counter_intuitive"]).mean()

np.float64(0.004707532051282051)

In [14]:
df_lichess_correct_ratings[df_lichess_correct_ratings["fen"].isna()]

,target_themes,target_rating,fen,is_legal,is_puzzle,counter_intuitive,actual_themes,predicted_rating,themes_match
4798,"['crushing', 'defensiveMove', 'enPassant', 'en...",1942.0,NaN,False,False,NaN,NaN,NaN,NaN
6147,"['crushing', 'defensiveMove', 'endgame', 'skew...",1920.0,NaN,False,False,NaN,NaN,NaN,NaN


In [7]:
dataset = pd.read_csv("./dataset/dataset.csv", chunksize=100_000)
external_puzzles = set()
for chunk in dataset:
    clean_puzzles = chunk["Puzzle_FEN"].str.split(" ").str[:4].str.join(" ")
    external_puzzles.update(clean_puzzles)
print(f"Total unique FENs in external dataset: {len(external_puzzles)}")
processed_lichess_fens = df_lichess_correct_ratings["fen"].fillna("").str.split(" ").str[:4].str.join(" ")
df_lichess_correct_ratings["is_overlap"] = processed_lichess_fens.isin(external_puzzles)
print(df_lichess_correct_ratings["is_overlap"].value_counts())


Total unique FENs in external dataset: 5565407
is_overlap
False    9972
True       12
Name: count, dtype: int64


In [ ]:
external_list = list(external_puzzles)
unique_lichess_fens = processed_lichess_fens.unique()[:100]

min_distances = {}

print(f"Computing distances for {len(unique_lichess_fens)} unique Lichess positions...")

for i, l_fen in tqdm(enumerate(unique_lichess_fens)):
    if not l_fen:
        continue
    
    minimum = min(Levenshtein.distance(l_fen, e_fen) for e_fen in external_list)
    # print(i, minimum)
    min_distances[l_fen] = minimum

df_lichess_correct_ratings["mean_edit_distance"] = processed_lichess_fens.map(min_distances)

avg_dist = df_lichess_correct_ratings["mean_edit_distance"].mean()
print(f"Average min Edit Distance: {avg_dist}")

Computing distances for 100 unique Lichess positions...


100it [06:30,  3.90s/it]


Average min Edit Distance: 13.19


In [9]:
import random


external_list = list(external_puzzles)
sample_size = 100
random_sample_fens = random.sample(external_list, sample_size)

min_distances = {}

print(f"Computing minimum nonzero distances for {sample_size} random positions from external_list...")

for s_fen in tqdm(random_sample_fens):
    if not s_fen:
        continue
    
    distances = (Levenshtein.distance(s_fen, e_fen) for e_fen in external_list)
    nonzero_distances = (d for d in distances if d > 0)
    
    minimum = min(nonzero_distances)
    min_distances[s_fen] = minimum

avg_dist = sum(min_distances.values()) / len(min_distances)
print(f"Average min nonzero Edit Distance: {avg_dist}")


Computing minimum nonzero distances for 100 random positions from external_list...


100%|██████████| 100/100 [06:33<00:00,  3.94s/it]


Average min nonzero Edit Distance: 13.12


In [25]:
df[df["is_legal"] & df["is_puzzle"] & df["counter_intuitive"]]

,target_themes,target_rating,fen,is_legal,is_puzzle,counter_intuitive,actual_themes,predicted_rating,themes_match
47,"['veryLong', 'endgame', 'queenEndgame', 'crush...",2884.556641,6k1/pp1Q1pp1/2p1p2p/2P1P1PP/1P3P2/3K4/8/7q w -...,True,True,True,"['endgame', 'mateIn3', 'mate', 'queenEndgame',...",0.398133,False


In [18]:
df_lichess[df_lichess["is_legal"] & df_lichess["is_puzzle"] & df_lichess["counter_intuitive"]]

,target_themes,target_rating,fen,is_legal,is_puzzle,counter_intuitive,actual_themes,predicted_rating,themes_match
238,"['bishopEndgame', 'crushing', 'defensiveMove',...",125.0,8/7K/6p1/8/6k1/3p4/3B4/8 b - - 0 53,True,True,True,"['endgame', 'advantage', 'advancedPawn', 'quie...",1788.979980,False
350,"['mate', 'mateIn2', 'middlegame', 'short']",24.0,3r2k1/pp1q3p/6p1/2Q1RnN1/5R2/P5Pb/1P2P2P/6K1 b...,True,True,True,"['middlegame', 'mateIn2', 'mate', 'short']",1050.020508,True


In [ ]:
with pd.option_context("display.max_colwidth", 500):
    display(df_lichess_large[df_lichess_large["is_legal"] & df_lichess_large["is_puzzle"] & df_lichess_large["counter_intuitive"]][["fen", "target_themes", "actual_themes", "predicted_rating"]])

,fen,target_themes,actual_themes,predicted_rating
1022,N4k2/3n2r1/pp1P1p1n/5Pqp/2P2pp1/1P6/P3RPPP/4R1K1 w - - 0 30,"['fork', 'mate', 'mateIn2', 'middlegame', 'short']","['middlegame', 'mateIn2', 'mate', 'short']",626.755493
3869,4k3/Rp1n1qpr/2p1r3/B7/2p5/2P4P/2P1QbP1/4R2K w - - 0 29,"['advantage', 'fork', 'long', 'middlegame']","['middlegame', 'advantage', 'exposedKing', 'long']",1872.460571
5814,7k/1b2r1p1/5r2/p1pp4/3q1b2/P1N5/1P2QPPP/1B2R1K1 w - - 0 28,"['long', 'mate', 'mateIn3', 'middlegame', 'sacrifice']","['middlegame', 'advantage', 'discoveredAttack', 'kingsideAttack', 'short']",1411.959717
7458,6k1/p2nr1Pr/1p5p/1b1p4/3P3q/4PR2/PP3RPP/5QK1 w - - 1 29,"['advancedPawn', 'mate', 'mateIn2', 'middlegame', 'promotion', 'short']","['middlegame', 'mateIn3', 'mate', 'exposedKing', 'kingsideAttack', 'long']",1248.470703
7710,r3r1k1/pp1b1ppp/8/q2pR3/5N2/1P6/1P2QPPP/4R1K1 w - - 0 20,"['advantage', 'fork', 'middlegame', 'veryLong']","['middlegame', 'mateIn3', 'mate', 'backRankMate', 'long']",775.752197
9797,6kq/p1pr1p1p/1bp3pP/8/4N3/6Pb/PPP2P2/4R1KR w - - 0 24,"['discoveredAttack', 'mate', 'mateIn2', 'middlegame', 'short']","['middlegame', 'mateIn2', 'mate', 'sacrifice', 'clearance', 'kingsideAttack', 'short']",1005.286377


In [8]:
with pd.option_context("display.max_colwidth", 500):
    display(df_lichess_correct_ratings[df_lichess_correct_ratings["is_legal"] & df_lichess_correct_ratings["is_puzzle"] & df_lichess_correct_ratings["counter_intuitive"]][["fen", "target_themes", "actual_themes", "predicted_rating"]])

,fen,target_themes,actual_themes,predicted_rating
219,r1bq1r1k/p5bp/1pn1Q1p1/n1p3N1/P1B2P2/2PP4/6PP/R1B2RK1 w - - 5 18,"['kingsideAttack', 'master', 'mate', 'mateIn2', 'middlegame', 'sacrifice', 'short', 'smotheredMate']","['middlegame', 'mateIn2', 'mate', 'smotheredMate', 'sacrifice', 'kingsideAttack', 'short']",1534.806641
229,8/5p1p/2k2Bp1/8/3p1PP1/6KP/1bP5/8 b - - 1 36,"['bishopEndgame', 'crushing', 'endgame', 'short', 'skewer']","['endgame', 'advantage', 'discoveredAttack', 'bishopEndgame', 'short']",1107.149170
384,5rk1/5qpp/4p3/3pQ3/3P4/6R1/PPr2PPP/5RK1 b - - 1 25,"['endgame', 'master', 'mate', 'mateIn4', 'sacrifice', 'veryLong']","['endgame', 'mateIn4', 'mate', 'sacrifice', 'veryLong']",1475.327026
401,r1bq1rk1/1pp2pp1/2nb1n2/p2Np1Np/3P3P/4P3/PPQ2PP1/R1B1KB1R w KQ - 0 11,"['kingsideAttack', 'mate', 'mateIn2', 'opening', 'short']","['middlegame', 'mateIn2', 'mate', 'kingsideAttack', 'short']",1221.063110
419,r4r1k/pp2Nppp/4n3/3p3R/8/2P1R3/PP6/2K5 w - - 1 25,"['anastasiaMate', 'attraction', 'endgame', 'long', 'mate', 'mateIn3', 'sacrifice']","['endgame', 'mateIn2', 'mate', 'anastasiaMate', 'attraction', 'sacrifice', 'short']",1389.271729
657,r1bqrbk1/1p1p1R1p/p1p3p1/4PN1n/2Bn1P2/2N5/PPP2P1P/R3K2R w KQ - 0 18,"['doubleCheck', 'kingsideAttack', 'mate', 'mateIn2', 'middlegame', 'short']","['middlegame', 'mateIn2', 'mate', 'doubleCheck', 'kingsideAttack', 'short']",1997.696777
740,r2q1r1k/1b3p2/1p2pNnb/3pP3/8/pPp1P1P1/P1P1BPK1/3R3R w - - 0 24,"['hookMate', 'kingsideAttack', 'mate', 'mateIn2', 'middlegame', 'short']","['middlegame', 'mateIn2', 'mate', 'hookMate', 'hangingPiece', 'kingsideAttack', 'short']",1096.088745
749,2R2r1k/pp5p/4Q1n1/4p3/4P2P/6PB/P4P1K/3q1r2 w - - 11 31,"['attraction', 'long', 'mate', 'mateIn3', 'middlegame', 'sacrifice']","['middlegame', 'mateIn2', 'mate', 'pin', 'short']",1541.887451
1035,6k1/5p2/p4B1p/1pbpQ3/8/2PB3P/5qbK/8 w - - 0 30,"['advantage', 'deflection', 'endgame', 'master', 'veryLong']","['endgame', 'mateIn4', 'mate', 'attraction', 'deflection', 'sacrifice', 'fork', 'veryLong']",2009.621704
1154,2r2r2/1p3pkp/p4np1/2P1Q3/5N2/3BP2P/3q1PP1/5RK1 w - - 0 26,"['advantage', 'middlegame', 'short']","['middlegame', 'advantage', 'deflection', 'sacrifice', 'long']",1913.974365
